In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
genres_df = spark.read.format("delta").load(f"{silver_folder_path}/movie_genres")
crew_df = spark.read.format("delta").load(f"{silver_folder_path}/crew")
cast_df = spark.read.format("delta").load(f"{silver_folder_path}/cast")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
genres_df.printSchema()
crew_df.printSchema()
cast_df.printSchema()

In [0]:
import pyspark.sql.functions as F

df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(crew_df, crew_df.id == movies_metadata_df.id, "inner")
    .join(genres_df, genres_df.id == links_df.movie_id, "inner")
    .join(cast_df, cast_df.id == movies_metadata_df.id, "inner")
    .drop(
        links_df.movie_id,
        crew_df.crew_id,
        crew_df.crew_department,
        crew_df.id,
        cast_df.id,
        cast_df.cast_id,
    )
    # .filter("title = 'The Shawshank Redemption'")
    .filter("crew_job = 'Director'")
    .filter(cast_df.cast_order < 2)
    .groupBy(
        F.col("movie_id"),
        F.col("title"),
        F.col("runtime"),
        F.col("revenue"),
        F.col("collection_id"),
        F.col("collection_name"),
        F.col("crew_name"),
        F.col("cast_name"),
        F.col("cast_character"),
        F.col("genres_name"),
        F.col("overview"),
    )
    .agg(
        F.avg(F.col("rating")).alias("average_rating"),
        F.count(F.col("rating")).alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 50")
    .withColumnRenamed("crew_name", "director")
    .orderBy(F.col("average_rating").desc())
)
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save(f"{gold_folder_path}/movies")